In [1]:
import pandas as pd
import re

In [2]:
tmdb = pd.read_csv("/100_Days_of_ml/Hybrid_Movie_Recommendation_System/data/tmdb_5000_movies.csv")

movielens = pd.read_csv("/100_Days_of_ml/Hybrid_Movie_Recommendation_System/data/movies.csv")

In [4]:
tmdb.head()
movielens.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
tmdb["title"].head()

0                                      Avatar
1    Pirates of the Caribbean: At World's End
2                                     Spectre
3                       The Dark Knight Rises
4                                 John Carter
Name: title, dtype: object

In [6]:
movielens["title"].head()

0                      Toy Story (1995)
1                        Jumanji (1995)
2               Grumpier Old Men (1995)
3              Waiting to Exhale (1995)
4    Father of the Bride Part II (1995)
Name: title, dtype: object

In [7]:
def clean_title(title):
    title = re.sub(r"\(\d{4}\)","",title)
    title = title.strip()
    return title

In [8]:
tmdb["clean_title"] = tmdb["title"].apply(clean_title)

In [9]:
movielens['clean_title'] = movielens["title"].apply(clean_title)

In [10]:
# verify for tmdb
tmdb[["title","clean_title"]].head()

,title,clean_title
0,Avatar,Avatar
1,Pirates of the Caribbean: At World's End,Pirates of the Caribbean: At World's End
2,Spectre,Spectre
3,The Dark Knight Rises,The Dark Knight Rises
4,John Carter,John Carter


In [11]:
# verify for movie lens
movielens[["title","clean_title"]].head()

,title,clean_title
0,Toy Story (1995),Toy Story
1,Jumanji (1995),Jumanji
2,Grumpier Old Men (1995),Grumpier Old Men
3,Waiting to Exhale (1995),Waiting to Exhale
4,Father of the Bride Part II (1995),Father of the Bride Part II


In [12]:
common_movies = pd.merge(
                tmdb,
                movielens,
                on = "clean_title",
                how = "inner"
                )

In [13]:
common_movies.shape

(2786, 24)

In [15]:
common_movies.columns

Index(['budget', 'genres_x', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'clean_title', 'movieId', 'title_y', 'genres_y'],
      dtype='object')

In [17]:
common_movies[
    [
        "title_x",
        "title_y",
        "clean_title"
    ]
].head(5)

,title_x,title_y,clean_title
0,Avatar,Avatar (2009),Avatar
1,Pirates of the Caribbean: At World's End,Pirates of the Caribbean: At World's End (2007),Pirates of the Caribbean: At World's End
2,Spectre,Spectre (2015),Spectre
3,John Carter,John Carter (2012),John Carter
4,Spider-Man 3,Spider-Man 3 (2007),Spider-Man 3


In [18]:
common_movies = common_movies[
    [
        "id",
        "movieId",
        "title_x",
        "title_y",
        "clean_title"
    ]
]

common_movies.head()

,id,movieId,title_x,title_y,clean_title
0,19995,72998,Avatar,Avatar (2009),Avatar
1,285,53125,Pirates of the Caribbean: At World's End,Pirates of the Caribbean: At World's End (2007),Pirates of the Caribbean: At World's End
2,206647,136020,Spectre,Spectre (2015),Spectre
3,49529,93363,John Carter,John Carter (2012),John Carter
4,559,52722,Spider-Man 3,Spider-Man 3 (2007),Spider-Man 3


In [19]:
import re

def clean_title(title):

    # Remove year
    title = re.sub(r"\(\d{4}\)", "", title)

    # Convert to lowercase
    title = title.lower()

    # Replace punctuation with spaces
    title = re.sub(r"[^a-z0-9\s]", " ", title)

    # Remove multiple spaces
    title = re.sub(r"\s+", " ", title)

    # Remove leading and trailing spaces
    title = title.strip()

    return title

In [20]:
tmdb["clean_title"] = tmdb["title"].apply(clean_title)

movielens["clean_title"] = movielens["title"].apply(clean_title)

In [21]:
common_movies = pd.merge(
    tmdb,
    movielens,
    on="clean_title",
    how="inner"
)

In [22]:
common_movies = common_movies[
    [
        "id",
        "movieId",
        "title_x",
        "title_y",
        "clean_title"
    ]
]

In [23]:
common_movies = common_movies.rename(
    columns={
        "id": "tmdb_id",
        "movieId": "movielens_id",
        "title_x": "tmdb_title",
        "title_y": "movielens_title"
    }
)

In [24]:
common_movies.head()

,tmdb_id,movielens_id,tmdb_title,movielens_title,clean_title
0,19995,72998,Avatar,Avatar (2009),avatar
1,285,53125,Pirates of the Caribbean: At World's End,Pirates of the Caribbean: At World's End (2007),pirates of the caribbean at world s end
2,206647,136020,Spectre,Spectre (2015),spectre
3,49529,93363,John Carter,John Carter (2012),john carter
4,559,52722,Spider-Man 3,Spider-Man 3 (2007),spider man 3


In [25]:
import pickle

pickle.dump(
    common_movies,
    open("/100_Days_of_ml/Hybrid_Movie_Recommendation_System/models/movie_mapping.pkl", "wb")
)

In [27]:
import pickle

movie_mapping = pickle.load(
    open("/100_Days_of_ml/Hybrid_Movie_Recommendation_System/models/movie_mapping.pkl", "rb")
)

In [28]:
movie_mapping.head()

,tmdb_id,movielens_id,tmdb_title,movielens_title,clean_title
0,19995,72998,Avatar,Avatar (2009),avatar
1,285,53125,Pirates of the Caribbean: At World's End,Pirates of the Caribbean: At World's End (2007),pirates of the caribbean at world s end
2,206647,136020,Spectre,Spectre (2015),spectre
3,49529,93363,John Carter,John Carter (2012),john carter
4,559,52722,Spider-Man 3,Spider-Man 3 (2007),spider man 3
